In [1]:

#"""Neural adaptive PID control of a thrust AMB using the Cho et al. model.

#Reference
#---------
#Y. M. Cho, S. Srinivasan, J.-H. Oh and H. S. Kim, "Modelling and system
#identification of active magnetic bearing systems", Mathematical and Computer
#Modelling of Dynamical Systems, 13(2), 125-142, 2007.

#Cho et al. give the bias-linearized actuator and PWM transconductance amplifier
#in their Eqs. (13)-(14). With the user-requested reduced axial model, the plant
#implemented here is exactly

#                  k_i K_amp
# G(s)=X/V_sp = --------------------------- .
#                (m s^2-k_x)(tau s+1)

#The state realization is
#    x_dot = v
#    m v_dot = k_x*x + k_i*i_c + F_disturbance
#    tau i_c_dot = -i_c + K_amp*V_sp.

#IMPORTANT SIGN CONVENTION: ``k_x`` below is the positive magnitude of the
#destabilizing (negative mechanical stiffness) term in ``m*x_ddot-k_x*x``.
#Cho et al. describe their actuator coefficient k_x as signed and negative.
#Convert an experimentally reported signed coefficient to the convention used
#here before entering it.
#
#The script contains an automatic RK4-versus-analytical-transfer-function test,
#a noisy filtered sensor, digital delay, PWM/voltage/current limits, anti-windup,
#an online neural gain scheduler, explicit disturbance injection, plots and
#validation metrics. It is a small-signal bias-linearized model; x is axial
#displacement from the nominal gap, not the absolute gap itself.

#Dependencies: numpy, scipy, matplotlib
#Run: python Cho2007_Neural_Adaptive_PID_Thrust_AMB.py
#"""
from __future__ import annotations

import argparse
import json
from collections import deque
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Callable, Deque, Dict, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
from scipy import signal


@dataclass
class AMBPlantParameters:
#    """All replaceable physical and implementation parameters in SI units."""

    # Bias-linearized axial plant: m*x_ddot-kx*x=ki*ic+F_d.
    rotor_mass: float = 0.42                    # m [kg]
    negative_stiffness_magnitude: float = 120.0 # k_x [N/m], enter POSITIVE magnitude
    current_force_gain: float = 3.8             # k_i [N/A]
    amplifier_gain: float = 0.45                # K_amp [A/V]
    amplifier_time_constant: float = 0.006      # tau [s]
    nominal_air_gap: float = 0.014              # [m]
    bias_current: float = 2.20                   # physical DC bias [A]

    # Constraints. ic is incremental control current about bias_current.
    control_current_min: float = -1.80           # [A]
    control_current_max: float = 1.80            # [A]
    voltage_setpoint_limit: float = 10.0         # |V_sp| [V]
    dc_bus_voltage: float = 48.0                 # PWM DC link [V]
    pwm_duty_limit: float = 0.95                 # absolute duty ratio
    amplifier_efficiency: float = 0.97

    # Position safety region expressed about nominal_air_gap.
    minimum_air_gap: float = 0.010               # [m]
    maximum_air_gap: float = 0.018               # [m]
    initial_displacement: float = 0.25e-3        # x(0) [m]
    initial_velocity: float = 0.0                # [m/s]

    def effective_voltage_limit(self) -> float:
        return min(self.voltage_setpoint_limit,
                   self.dc_bus_voltage*self.pwm_duty_limit)

    def analytical_tf(self) -> Tuple[np.ndarray, np.ndarray]:
#        """Numerator/denominator of the exact requested G(s)."""
        m, kx, tau = (self.rotor_mass, self.negative_stiffness_magnitude,
                      self.amplifier_time_constant)
        numerator = np.array([self.current_force_gain*self.amplifier_gain])
        # (m*s^2-kx)*(tau*s+1)
        denominator = np.array([m*tau, m, -kx*tau, -kx])
        return numerator, denominator


@dataclass
class SensorParameters:
    sensitivity: float = 800.0       # [V/m]
    bias_voltage: float = 0.0008     # residual uncompensated offset [V]
    noise_variance: float = 4.0e-7   # voltage-noise variance [V^2]
    lowpass_cutoff_hz: float = 100.0 # 2nd-order Butterworth anti-alias filter


@dataclass
class NeuralPIDParameters:
    sample_time: float = 0.0005
    computational_delay_samples: int = 1
    hidden_neurons: int = 10
    learning_rate: float = 1.0e-4
    weight_decay: float = 1.0e-5
    gradient_clip_norm: float = 4.0
    adaptation_deadband: float = 8.0e-6
    jacobian_horizon: float = 0.020
    error_scale: float = 0.50e-3
    derivative_scale: float = 0.050
    integral_scale: float = 0.002
    derivative_filter_tau: float = 0.003
    antiwindup_gain: float = 8.0
    integrator_limit: float = 0.004
    # [Kp, Ki, Kd] bounds in V/m, V/(m*s), V*s/m.
    gain_min: Tuple[float, float, float] = (150.0, 500.0, 5.0)
    gain_max: Tuple[float, float, float] = (1500.0, 7000.0, 100.0)
    gain_initial: Tuple[float, float, float] = (500.0, 1800.0, 20.0)


class ThrustAMBPlant:
#    """Saturated RK4 realization of the requested third-order transfer model."""

    def __init__(self, p: AMBPlantParameters):
        self.p = p
        self.state = np.array([p.initial_displacement, p.initial_velocity, 0.0])
        self.voltage_applied = 0.0
        self.duty_ratio = 0.0

    def _limited_voltage(self, voltage_demand: float) -> float:
        p = self.p
        voltage = float(np.clip(voltage_demand, -p.effective_voltage_limit(),
                                p.effective_voltage_limit()))
        self.duty_ratio = voltage/p.dc_bus_voltage
        return p.amplifier_efficiency*voltage

    def derivative(self, state: np.ndarray, voltage_demand: float,
                   disturbance_force: float, apply_limits: bool = True) -> np.ndarray:
        p = self.p
        x, velocity, incremental_current = state
        voltage = self._limited_voltage(voltage_demand) if apply_limits else voltage_demand
        current_target = p.amplifier_gain*voltage
        if apply_limits:
            current_target = float(np.clip(current_target,p.control_current_min,
                                           p.control_current_max))
        current_dot = (current_target-incremental_current)/p.amplifier_time_constant
        # Stop integration from driving an already-saturated current farther out.
        if apply_limits and ((incremental_current <= p.control_current_min and current_dot < 0)
                             or (incremental_current >= p.control_current_max and current_dot > 0)):
            current_dot = 0.0
        acceleration = (p.negative_stiffness_magnitude*x
                        +p.current_force_gain*incremental_current
                        +disturbance_force)/p.rotor_mass
        return np.array([velocity,acceleration,current_dot])

    def step(self, voltage_demand: float, disturbance_force: float,
             dt: float, substeps: int = 4) -> None:
#        """Classical fixed-step RK4 within one digital-control interval."""
        h = dt/substeps
        for _ in range(substeps):
            s = self.state
            k1 = self.derivative(s,voltage_demand,disturbance_force)
            k2 = self.derivative(s+0.5*h*k1,voltage_demand,disturbance_force)
            k3 = self.derivative(s+0.5*h*k2,voltage_demand,disturbance_force)
            k4 = self.derivative(s+h*k3,voltage_demand,disturbance_force)
            self.state = s+h*(k1+2*k2+2*k3+k4)/6.0
        self.state[2] = np.clip(self.state[2],self.p.control_current_min,
                                self.p.control_current_max)
        self.voltage_applied = self._limited_voltage(voltage_demand)


class SensorFilter:
#    """Sensitivity, residual bias, Gaussian noise and Butterworth filtering."""

    def __init__(self, p: SensorParameters, sample_time: float,
                 initial_displacement: float, seed: int = 21):
        self.p = p
        self.rng = np.random.default_rng(seed)
        k = np.tan(np.pi*p.lowpass_cutoff_hz*sample_time)
        norm = 1.0/(1.0+np.sqrt(2.0)*k+k*k)
        self.b = np.array([k*k*norm,2*k*k*norm,k*k*norm])
        self.a = np.array([1.0,2*(k*k-1)*norm,
                           (1-np.sqrt(2.0)*k+k*k)*norm])
        initial_v = p.sensitivity*initial_displacement+p.bias_voltage
        self.input_history = np.array([initial_v,initial_v])
        self.output_history = np.array([initial_v,initial_v])

    def measure(self, displacement: float) -> float:
        sensor_voltage = (self.p.sensitivity*displacement+self.p.bias_voltage
                          +self.rng.normal(0.0,np.sqrt(self.p.noise_variance)))
        y = (self.b[0]*sensor_voltage+self.b[1]*self.input_history[0]
             +self.b[2]*self.input_history[1]-self.a[1]*self.output_history[0]
             -self.a[2]*self.output_history[1])
        self.input_history[1],self.input_history[0] = self.input_history[0],sensor_voltage
        self.output_history[1],self.output_history[0] = self.output_history[0],y
        # Divide by sensitivity; residual bias remains visible to the controller.
        return y/self.p.sensitivity


class NeuralAdaptivePIDController:
#    """Online NN scheduler for strictly bounded positive PID gains.

#    Input: z=[e/e_s, e_dot/d_s, integral(e)/i_s].
#    Hidden layer: tanh(W1*z+b1).
#    Output: sigmoid(W2*h+b2), affinely mapped to positive gain intervals.

#    The update uses a delayed sensitivity cache because G(s) has relative
#    degree three. For J=0.5*(e/e_s)^2, e=r-y, and plant sensitivity
#    G_J=dy/du evaluated over ``jacobian_horizon``, the chain rule is

#      dJ/dw = -(e/e_s^2)*G_J * [e, integral(e), e_dot]
#               * d[Kp,Ki,Kd]/d(network_output) * d(network_output)/dw.

#    ``G_J`` is not guessed: it is obtained by RK4 integration of the exact
#    sensitivity realization derived from the requested transfer function with
#    a unit voltage step. The cache from that horizon ago is updated, producing
#   a practical truncated-BPTT/instantaneous-gradient law.
#    """

    def __init__(self, p: NeuralPIDParameters, plant_p: AMBPlantParameters,
                 seed: int = 7):
        self.p,self.plant_p = p,plant_p
        rng = np.random.default_rng(seed)
        self.W1 = rng.normal(0.0,0.10,(p.hidden_neurons,3))
        self.b1 = np.zeros(p.hidden_neurons)
        self.W2 = rng.normal(0.0,0.012,(3,p.hidden_neurons))
        lower,upper,initial = map(np.asarray,(p.gain_min,p.gain_max,p.gain_initial))
        q = np.clip((initial-lower)/(upper-lower),1e-6,1-1e-6)
        self.b2 = np.log(q/(1-q))
        self.integral = 0.0
        self.derivative = 0.0
        self.previous_error = 0.0
        self.first_sample = True
        self.plant_jacobian = self._transfer_function_jacobian()
        delay = max(1,int(round(p.jacobian_horizon/p.sample_time)))
        self.cache: Deque[Optional[Dict[str,np.ndarray]]] = deque([None]*delay,maxlen=delay)

    @staticmethod
    def _sigmoid(x: np.ndarray) -> np.ndarray:
        return 1.0/(1.0+np.exp(-np.clip(x,-35.0,35.0)))

    def _transfer_function_jacobian(self) -> float:
#        """Finite-horizon dy/du from the exact G(s) sensitivity dynamics."""
        p,dt = self.plant_p,self.p.sample_time/4.0
        state = np.zeros(3)
        steps = int(round(self.p.jacobian_horizon/dt))

        def f(s):
            x,v,i = s
            return np.array([v,(p.negative_stiffness_magnitude*x
                                +p.current_force_gain*i)/p.rotor_mass,
                             (p.amplifier_gain-i)/p.amplifier_time_constant])
        for _ in range(steps):
            k1=f(state); k2=f(state+0.5*dt*k1); k3=f(state+0.5*dt*k2); k4=f(state+dt*k3)
            state += dt*(k1+2*k2+2*k3+k4)/6.0
        return float(state[0])

    def _forward(self,z: np.ndarray) -> Tuple[np.ndarray,Dict[str,np.ndarray]]:
        hidden = np.tanh(self.W1@z+self.b1)
        output = self._sigmoid(self.W2@hidden+self.b2)
        lower,upper = np.asarray(self.p.gain_min),np.asarray(self.p.gain_max)
        gains = lower+(upper-lower)*output
        return gains,{'z':z,'h':hidden,'o':output}

    def _adapt_from_delayed_cache(self,current_error: float) -> None:
        old = self.cache[0]
        if old is None or abs(current_error)<self.p.adaptation_deadband:
            return
        p = self.p
        # de/du = -dy/du because e=r-y.
        dJ_du = -(current_error/p.error_scale**2)*self.plant_jacobian
        gain_range = np.asarray(p.gain_max)-np.asarray(p.gain_min)
        delta_output = dJ_du*old['pid_basis']*gain_range*old['o']*(1-old['o'])
        grad_W2 = np.outer(delta_output,old['h'])
        grad_b2 = delta_output
        delta_hidden = (self.W2.T@delta_output)*(1-old['h']**2)
        grad_W1 = np.outer(delta_hidden,old['z'])
        grad_b1 = delta_hidden
        norm = np.sqrt(np.sum(grad_W1**2)+np.sum(grad_W2**2)
                       +np.sum(grad_b1**2)+np.sum(grad_b2**2))
        scale = min(1.0,p.gradient_clip_norm/max(norm,1e-12))
        self.W1 -= p.learning_rate*(scale*grad_W1+p.weight_decay*self.W1)
        self.W2 -= p.learning_rate*(scale*grad_W2+p.weight_decay*self.W2)
        self.b1 -= p.learning_rate*scale*grad_b1
        self.b2 -= p.learning_rate*scale*grad_b2

    def compute(self,reference: float,measured_displacement: float,
                voltage_limit: float) -> Tuple[float,np.ndarray,float]:
        p,dt = self.p,self.p.sample_time
        error = reference-measured_displacement
        raw_derivative = 0.0 if self.first_sample else (error-self.previous_error)/dt
        alpha = dt/(p.derivative_filter_tau+dt)
        self.derivative += alpha*(raw_derivative-self.derivative)
        self._adapt_from_delayed_cache(error)
        z = np.array([error/p.error_scale,self.derivative/p.derivative_scale,
                      self.integral/p.integral_scale])
        z = np.clip(z,-5.0,5.0)
        gains,cache = self._forward(z)
        basis = np.array([error,self.integral,self.derivative])
        voltage_raw = float(gains@basis)
        voltage = float(np.clip(voltage_raw,-voltage_limit,voltage_limit))

        # Back-calculation anti-windup: saturation error is mapped back to the
        # integrator input through Ki, then integrated at the controller rate.
        antiwindup = p.antiwindup_gain*(voltage-voltage_raw)/max(gains[1],1e-9)
        self.integral = float(np.clip(self.integral+dt*(error+antiwindup),
                                      -p.integrator_limit,p.integrator_limit))
        cache['pid_basis']=basis
        self.cache.append(cache)
        self.previous_error,self.first_sample = error,False
        return voltage,gains,error


class SimulationEnvironment:
    def __init__(self,plant: ThrustAMBPlant,sensor: SensorFilter,
                 controller: NeuralAdaptivePIDController,duration: float=3.0):
        self.plant,self.sensor,self.controller = plant,sensor,controller
        self.duration,self.ts = duration,controller.p.sample_time
        n=max(0,controller.p.computational_delay_samples)
        self.delay_line=deque([0.0]*(n+1),maxlen=n+1)

    @staticmethod
    def default_disturbance(t: float) -> float:
#        """Step at 0.5 s, sinusoidal surge after 1.5 s and finite pulse."""
        step=0.55 if t>=0.5 else 0.0
        sine=0.10*np.sin(2*np.pi*1.2*(t-1.5)) if t>=1.5 else 0.0
        pulse=-0.30 if 2.15<=t<2.22 else 0.0
        return step+sine+pulse

    def run(self,disturbance: Optional[Callable[[float],float]]=None) -> Dict[str,np.ndarray]:
        disturbance=disturbance or self.default_disturbance
        n=int(round(self.duration/self.ts))+1
        names=('time','displacement','air_gap','measured_displacement','velocity',
               'incremental_current','total_current','voltage_demand','voltage_applied',
               'pwm_duty','disturbance','kp','ki','kd','error')
        log={name:np.zeros(n) for name in names}
        voltage_limit=self.plant.p.effective_voltage_limit()
        for k in range(n):
            t=k*self.ts
            x,v,ic=self.plant.state
            measured=self.sensor.measure(x)
            voltage,gains,error=self.controller.compute(0.0,measured,voltage_limit)
            delayed=self.delay_line.popleft(); self.delay_line.append(voltage)
            force=disturbance(t)
            values=(t,x,self.plant.p.nominal_air_gap+x,measured,v,ic,
                    self.plant.p.bias_current+ic,voltage,self.plant.voltage_applied,
                    self.plant.duty_ratio,force,*gains,error)
            for name,value in zip(names,values): log[name][k]=value
            if k<n-1: self.plant.step(delayed,force,self.ts)
        return log


def validate_transfer_function(p: AMBPlantParameters,dt: float=1e-4,
                               duration: float=0.12) -> Dict[str,float]:
#    """Compare unsaturated RK4 state equations with scipy.signal G(s)."""
    t=np.arange(0.0,duration+0.5*dt,dt)
    u=0.08*np.sin(2*np.pi*7.0*t)+0.03*(t>=0.035)
    numerical=np.zeros_like(t); state=np.zeros(3)

    def f(s,uk):
        x,v,i=s
        return np.array([v,(p.negative_stiffness_magnitude*x+p.current_force_gain*i)/p.rotor_mass,
                         (p.amplifier_gain*uk-i)/p.amplifier_time_constant])
    for k in range(len(t)-1):
        h=dt; u0=u[k]; u1=u[k+1]; um=0.5*(u0+u1)
        k1=f(state,u0); k2=f(state+0.5*h*k1,um)
        k3=f(state+0.5*h*k2,um); k4=f(state+h*k3,u1)
        state += h*(k1+2*k2+2*k3+k4)/6.0
        numerical[k+1]=state[0]
    numerator,denominator=p.analytical_tf()
    _,analytical,_=signal.lsim(signal.TransferFunction(numerator,denominator),U=u,T=t)
    error=numerical-analytical
    scale=max(np.max(np.abs(analytical)),1e-12)
    return {'tf_validation_RMSE_m':float(np.sqrt(np.mean(error**2))),
            'tf_validation_normalized_RMSE_percent':float(100*np.sqrt(np.mean(error**2))/scale),
            'tf_validation_peak_error_m':float(np.max(np.abs(error)))}


def convergence_time(t: np.ndarray,x: np.ndarray,event: float=0.5,
                     tolerance: float=0.10e-3,dwell: float=0.15,
                     search_end: float=1.49) -> float:
    dt=t[1]-t[0]; width=max(1,int(round(dwell/dt)))
    start=int(np.searchsorted(t,event)); stop=min(int(np.searchsorted(t,search_end)),len(t)-width)
    inside=np.abs(x)<=tolerance
    for k in range(start,stop):
        if np.all(inside[k:k+width]): return float(t[k]-event)
    return float('nan')


def metrics(log: Dict[str,np.ndarray],p: AMBPlantParameters) -> Dict[str,float]:
    x=log['displacement']; gap=log['air_gap']
    return {'position_RMSE_um':float(1e6*np.sqrt(np.mean(x*x))),
            'peak_gap_deviation_um':float(1e6*np.max(np.abs(x))),
            'maximum_total_current_A':float(np.max(log['total_current'])),
            'maximum_incremental_current_A':float(np.max(np.abs(log['incremental_current']))),
            'maximum_voltage_demand_V':float(np.max(np.abs(log['voltage_demand']))),
            'convergence_time_after_step_s':convergence_time(log['time'],x),
            'minimum_air_gap_mm':float(1e3*np.min(gap)),
            'maximum_air_gap_mm':float(1e3*np.max(gap)),
            'safety_bound_violations':int(np.sum((gap<p.minimum_air_gap)|(gap>p.maximum_air_gap)))}


def plot_results(log: Dict[str,np.ndarray],p: AMBPlantParameters,path: Path) -> None:
    t=log['time']; fig,ax=plt.subplots(4,1,figsize=(11,10),sharex=True)
    ax[0].plot(t,1e3*log['air_gap'],label='actual air gap',lw=1.5)
    ax[0].axhline(1e3*p.nominal_air_gap,color='k',ls=':',label='reference')
    ax[0].axhspan(1e3*p.minimum_air_gap,1e3*p.maximum_air_gap,color='green',alpha=.08,label='safe region')
    ax[0].set_ylabel('Air gap [mm]'); ax[0].legend(ncol=3); ax[0].grid(alpha=.25)
    ax[1].plot(t,log['total_current'],label='total coil current')
    ax[1].plot(t,log['incremental_current'],'--',label='incremental current')
    axv=ax[1].twinx(); axv.plot(t,log['voltage_demand'],color='tab:red',alpha=.7,label='Vsp demand')
    lim=p.effective_voltage_limit(); axv.axhline(lim,color='tab:red',ls=':'); axv.axhline(-lim,color='tab:red',ls=':')
    ax[1].set_ylabel('Current [A]'); axv.set_ylabel('Voltage [V]',color='tab:red')
    ax[1].legend(loc='upper left'); axv.legend(loc='upper right'); ax[1].grid(alpha=.25)
    ax[2].plot(t,log['kp'],label=r'$K_p$'); ax[2].plot(t,log['ki'],label=r'$K_i$'); ax[2].plot(t,log['kd'],label=r'$K_d$')
    ax[2].set_ylabel('Adaptive PID gains'); ax[2].legend(ncol=3); ax[2].grid(alpha=.25)
    ax[3].plot(t,log['disturbance'],color='tab:purple',label='axial thrust disturbance')
    ax[3].set_ylabel('Load [N]'); ax[3].set_xlabel('Time [s]'); ax[3].legend(); ax[3].grid(alpha=.25)
    fig.suptitle('Cho et al. (2007) Transfer-Function AMB - Neural Adaptive PID')
    fig.tight_layout(); fig.savefig(path,dpi=180); plt.close(fig)


def main(argv=None) -> Dict[str,float]:
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--output-dir',default='cho2007_adaptive_pid_results')
    parser.add_argument('--duration',type=float,default=3.0)
    parser.add_argument('--seed',type=int,default=21)
    args,_=parser.parse_known_args(argv) # safe inside Jupyter (-f kernel.json)
    out=Path(args.output_dir); out.mkdir(parents=True,exist_ok=True)
    pp,sp,cp=AMBPlantParameters(),SensorParameters(),NeuralPIDParameters()
    tf_check=validate_transfer_function(pp)
    plant=ThrustAMBPlant(pp); sensor=SensorFilter(sp,cp.sample_time,pp.initial_displacement,args.seed)
    controller=NeuralAdaptivePIDController(cp,pp,args.seed+1)
    log=SimulationEnvironment(plant,sensor,controller,args.duration).run()
    report={**metrics(log,pp),**tf_check,'estimated_plant_jacobian_m_per_V':controller.plant_jacobian}
    plot_results(log,pp,out/'cho2007_neural_adaptive_pid.png')
    np.savez_compressed(out/'simulation_data.npz',**log)
    with open(out/'metrics.json','w',encoding='utf-8') as f: json.dump(report,f,indent=2)
    with open(out/'parameters.json','w',encoding='utf-8') as f:
        json.dump({'plant':asdict(pp),'sensor':asdict(sp),'controller':asdict(cp)},f,indent=2)
    print('\nCho et al. transfer-function Neural Adaptive PID validation')
    print('-'*62)
    for key,value in report.items():
        shown='not reached' if isinstance(value,float) and np.isnan(value) else f'{value:.6g}'
        print(f'{key:43s}: {shown}')
    print(f'Results directory                          : {out.resolve()}')
    return report


if __name__=='__main__':
    main()


Cho et al. transfer-function Neural Adaptive PID validation
--------------------------------------------------------------
position_RMSE_um                           : 187.754
peak_gap_deviation_um                      : 687.415
maximum_total_current_A                    : 2.23386
maximum_incremental_current_A              : 0.278942
maximum_voltage_demand_V                   : 0.67061
convergence_time_after_step_s              : 0.472
minimum_air_gap_mm                         : 13.5284
maximum_air_gap_mm                         : 14.6874
safety_bound_violations                    : 0
tf_validation_RMSE_m                       : 7.5706e-16
tf_validation_normalized_RMSE_percent      : 4.0469e-11
tf_validation_peak_error_m                 : 2.20388e-15
estimated_plant_jacobian_m_per_V           : 0.00047032
Results directory                          : C:\Users\User\cho2007_adaptive_pid_results
